In [ ]:
import chephy_model as cpm
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [3]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
# atchley_df = pd.read_csv(atchley_path)
# atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

downsample_size = 10000

downsampled_path = f"/dsi/efroni-lab/sbm/chen2/downsampled_data/{downsample_size}/"

graph_path = f"/dsi/efroni-lab/sbm/chen2/graph_data/{downsample_size}"

In [4]:
from pathlib import Path
file_names = [downsampled_path + f.name for f in Path(downsampled_path).iterdir() if f.is_file()]

In [5]:
# Example usage
from chephy_model import (
    load_atchley_map, build_patient_knn_graph, KNNConfig
)
from torch_geometric.data import Data

def get_graph_data(sequences, label, node_weights=None):

    # Build a binary k-NN graph (no sigma needed)
    edge_index, edge_weight, X, scaler = build_patient_knn_graph(
        sequences,
        k=10,
        atchley_map=load_atchley_map(),
        standardize_features=True,   # z-score the 40D features (recommended if you’ll train σ on train set)
        weighting="self_tuning",            # change to "rbf" or "self_tuning" for weighted edges
        mutual=True,                 # mutual k-NN is cleaner; set False for union
        knn_cfg=KNNConfig(k=10, use_ann_if_large=True, ann_threshold=5000),
    )

    data_dict = {
        'x': torch.tensor(X, dtype=torch.float32),
        'edge_index': torch.tensor(edge_index, dtype=torch.long),
    }
    
    edge_weight = torch.tensor(edge_weight, dtype=torch.float32)
    data_dict['edge_weights'] = edge_weight

    data_dict["y"] = torch.tensor([label])

    if node_weights is not None:
        node_weights_tensor = torch.tensor(node_weights, dtype=torch.float32)
        data_dict['node_weights'] = node_weights_tensor


    graph = Data(**data_dict).to(device)
    return graph

    # If you want weighted edges with a global σ learned on training:
    # sigma = median_distance_over_training  (compute once from training patients)
    # edge_index, edge_weight, X, seqs, _ = build_patient_knn_graph(..., weighting="rbf", sigma=sigma)

    


In [ ]:
from tqdm import tqdm
import os

cdr3_header = "aaSeqCDR3"


sub_folder = f"{graph_path}"

if not os.path.exists(sub_folder):
    os.makedirs(sub_folder)

for file_name in tqdm(file_names, desc="Processing files"):
    df = pd.read_csv(file_name, sep="\t")
    label = file_name.split("/")[-1].split("_")[0]
    file_save_name = f"{sub_folder}/{file_name.rsplit('/', 1)[-1]}.pt"

    if label == "sick":
        label = 1
    else:
        label = 0

    data = prepare_data(df, cdr3_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = list(set(data["cdr3_truncated"]))
    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences.index(x) if x in sequences else -1)
    full_sequences_list = data.sort_values("tcr_index")[cdr3_header].tolist()

    node_weights = data.groupby(data['tcr_index'])["readFraction"].sum().tolist()
    
    graph = get_graph_data(sequences, label, node_weights)
    
    torch.save((graph, data), file_save_name)


Processing files:   0%|          | 0/91 [00:00<?, ?it/s]

Processing files: 100%|██████████| 91/91 [02:30<00:00,  1.66s/it]


<class 'torch.Tensor'> torch.Size([9207, 40])
<class 'torch.Tensor'> torch.Size([9207])
<class 'torch.Tensor'> torch.Size([2, 25471])
<class 'torch.Tensor'> torch.Size([25471])
<class 'torch.Tensor'> torch.Size([1])
